<a href="https://colab.research.google.com/github/Alisherbakh/computer_vison/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Menu Detector!')

Menu Detector!


In [11]:
# --------------------------
# Import Libraries

# --------------------------
from google.colab import drive        # Colab'ga Google Drive'ni ulash uchun modulni
import torch                      # Asosiy PyTorch kutubxonasi (tensorlar, GPU va boshqalar)
import torch.nn as nn             # Neyron tarmoq qismlari: qatlamlar (Linear) va loss funksiyalari (CrossEntropyLoss); qisqa nomi "nn"
import torch.optim as optim       # Optimizerlar to'plami (SGD, Adam) — model parametrlarini yangilash uchun; qisqa nomi "optim"
import torchvision.transforms as transforms   # Rasmlarni modelga berishdan oldin o'zgartiruvchi (tayyorlovchi) vositalarni chaqiramiz
from torchvision.models import mobilenet_v2   # torchvision'ning tayyor modellar to'plamidan MobileNetV2 modelini chaqiramiz
from PIL import Image, UnidentifiedImageError   # Image → rasmlarni ochish/o'qish; UnidentifiedImageError → ochilmaydigan (buzilgan) rasm xatosi
from torch.utils.data import Dataset, DataLoader           # O'z datasetimizni yaratish uchun PyTorch'ning asosiy klassi
import os                                       # Papka va fayllar bilan ishlash (yo'llar, ro'yxatlar)
import numpy as np                              # Sonlar va massivlar bilan ishlash


In [3]:
drive.mount('/content/drive')         # Google Drive'ni /content/drive papkasiga ulaymiz → Mounted at /content/drive


Mounted at /content/drive


In [4]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'   # Dataset papkasining manzili
print('Dataset_path:', DATASET_PATH)                      # Manzilni ekranga chiqaramiz


CUSTOM_CLASS_MAPPING = {                  # Papka nomi → model o'rganadigan sinf nomi (lug'at), papkadagi maxsulotlarni berilgan label deb tanib oladi
    "hamburger": "hamburger",             # hamburger papkasi → "hamburger" sinfi
    "hot_dog": "hot_dog",                 # hot_dog papkasi → "hot_dog" sinfi
    "chocolate_cake": "dessert",  # label grouping | class consolidation   → shokoladli tort → "dessert" sinfiga qo'shiladi
    "cheesecake": "dessert",      # label grouping | class consolidation   → chizkeyk → ham "dessert" sinfiga qo'shiladi
    "kebab": "kebab",                     # kebab papkasi → "kebab" sinfi
    "pilaf": "pilaf"                      # pilaf papkasi → "pilaf" sinfi
}


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']   # Model ajratadigan 5 ta yakuniy sinf
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}          # Har bir sinfga raqam beramiz: nom → raqam
NUM_CLASSES = len(CLASSES)                                        # Sinflar soni → 5

print(NUM_CLASSES)                        # → 5
print(CLASS_TO_IDX)                       # → {'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}



transform = transforms.Compose([                          # Bir nechta transformni ketma-ket ulaymiz
    transforms.Resize((224, 224)),                        # Har qanday rasmni 224x224 o'lchamga keltiramiz
    transforms.ToTensor(),                                # Rasmni tensorga aylantiramiz: 0–255 → 0.0–1.0, H,W,C → C,H,W
    transforms.Normalize(mean=[0.485, 0.456, 0.406],      # Har bir kanal (R, G, B) uchun o'rtacha qiymat
                         std=[0.229, 0.224, 0.225])       # Har bir kanal (R, G, B) uchun standart og'ish
])

# 0 ~ 255                                                 → Asl rasmdagi piksel qiymati (butun son)
# 0.0 ~ 1.0                                               → ToTensor'dan keyin: 255 ga bo'linadi
# 0.234                                                   → Misol: bitta piksel qiymati ToTensor'dan keyin
# H, W, C => C, H, W | RGB = Red, Green, Blue             → O'lchamlar tartibi o'zgaradi: rang kanali oldinga o'tadi
# Normalize                                               → Sonlarni 0 atrofiga markazlashtiramiz
# pixel = (pixel - mean) / std                            → Normalize formulasi

Dataset_path: /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [5]:
# -----------------------
# Custom Dataset Class
# -----------------------

class FoodDataset(Dataset):                                   # O'z datasetimizni yaratamiz; PyTorch'ning Dataset klassidan meros olamiz
    def __init__(self, images, labels, transform=None):       # Konstruktor: dataset yaratilganda bir marta ishlaydi
        self.images = images                                  # Rasmlar yo'llari ro'yxatini saqlaymiz → ['.../pilaf/Image_1.jpg', ...]
        self.labels = labels                                  # Har bir rasmning sinf raqamini saqlaymiz → [4, 4, 3, ...]
        self.transform = transform                            # Rasmga qo'llanadigan transformni saqlaymiz (bo'lmasa None)

    def __len__(self):                                        # Datasetda nechta element borligini qaytaradi
        print('images_length', len(self.images))              # Rasmlar sonini ekranga chiqaramiz (tekshirish uchun)
        return len(self.images)                               # Rasmlar sonini qaytaramiz

    def __getitem__(self, idx):                               # idx-raqamli bitta rasm va uning labelini qaytaradi
        img_path = self.images[idx]                           # idx-chi rasmning fayl yo'lini olamiz
        print('image_path', img_path)                         # Yo'lni ekranga chiqaramiz (tekshirish uchun)
        label = self.labels[idx]                              # Shu rasmning labelini (sinf raqamini) olamiz
        print('label', label)                                 # Labelni ekranga chiqaramiz (tekshirish uchun)
        try:                                                  # Rasmni ochishga harakat qilamiz...
            image = Image.open(img_path).convert('RGB')   # 3 Channel   → rasmni ochib, 3 kanalli (RGB) formatga keltiramiz
        except (UnidentifiedImageError, OSError):             # ...agar rasm buzilgan bo'lsa yoki ochilmasa:
            print(f"Skipping broken image: {img_path}")       # Qaysi rasm buzilganini chiqaramiz
            return self.__getitem__((idx + 1) % len(self.images))   #   → keyingi rasmni olib qaytaramiz
        if self.transform:                                    # Agar transform berilgan bo'lsa...
            image = self.transform(image)                     # ...rasmni transformdan o'tkazamiz (Resize, ToTensor, Normalize)
        return image, label                                   # Tayyor rasm (tensor) va labelni juft qilib qaytaramiz

In [6]:
# ---------------------------
# Gather and Split Data
# ---------------------------
all_images = []                                                  # Barcha (rasm_yo'li, label) juftlarini yig'adigan bo'sh ro'yxat
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():   # Lug'atdagi har bir juftni olamiz: papka nomi → sinf nomi
    class_path = os.path.join(DATASET_PATH, original_class)   # /content/drive/MyDrive/food101_dataset/hamburger   → sinf papkasining to'liq yo'li
    print('class_path:', class_path)                             # Qaysi papka tekshirilayotganini chiqaramiz
    if not os.path.exists(class_path):                           # Agar bunday papka yo'q bo'lsa...
        print(f"Warning: {class_path} not found")                # ...ogohlantirish chiqaramiz
        continue                                                 # ...va keyingi sinfga o'tamiz (pastdagi kod ishlamaydi)
    for img in os.listdir(class_path):                           # Papkadagi har bir fayl nomini olamiz
        if img.endswith(('.jpg', '.jpeg', '.png')):   # .txt, .docx   → faqat rasm fayllarini olamiz, boshqalarini tashlab ketamiz
            full_path = os.path.join(class_path, img)   # /content/drive/MyDrive/food101_dataset/hamburger/100057.jpg   → rasmning to'liq yo'li
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))   # (yo'l, sinf raqami) juftini ro'yxatga qo'shamiz

np.random.shuffle(all_images)                                    # Ro'yxatni aralashtiramiz (sinflar tartib bilan turib qolmasligi uchun)
split = int(0.8 * len(all_images))                               # Umumiy sonning 80% i → bo'lish nuqtasi
train_data = all_images[:split]   # 1000 | 800 train_data | 200 val_data   → birinchi 80% → o'qitish uchun
val_data = all_images[split:]                                    # Qolgan 20% → tekshirish (validation) uchun

train_images, train_labels = zip(*train_data)                    # Juftlarni ikki alohida ro'yxatga ajratamiz: yo'llar va labellar
val_images, val_labels = zip(*val_data)                          # Validation uchun ham xuddi shunday

#print('all_images:', all_images)                                 # Barcha juftlarni chiqaramiz (tekshirish uchun)

dataset = FoodDataset(train_images, train_labels)                # O'qitish ma'lumotidan dataset obyektini yaratamiz
print(len(dataset))                                              # Datasetdagi rasmlar soni → __len__ ishlaydi
img, lbl = dataset[0]                                            # Birinchi rasm va labelini olamiz → __getitem__(0) ishlaydi

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
images_length 3316
3316
image_path /content/drive/MyDrive/food101_dataset/cheesecake/598947.jpg
label 2


In [7]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)   # O'qitish uchun dataset: 80% rasmlar + ularning labellari + transform
val_dataset = FoodDataset(val_images, val_labels, transform=transform)         # Tekshirish uchun dataset: 20% rasmlar + ularning labellari + transform

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)   #thread | parallel loading for speed   → o'qitish rasmlarini 32 tadan, aralashtirib, 2 ta yordamchi bilan yuklaydi
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)      # Tekshirish rasmlarini 32 tadan, aralashtirmasdan, 2 ta yordamchi bilan yuklaydi

images_length 3316
images_length 3316


In [9]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1")
# pretrained model | ligthweight | CNN | 1000 class | million   → ImageNet'da o'qitilgan tayyor MobileNetV2 modelini yuklaymiz
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
# fine-tuning | backbone | model layer freeze   → oxirgi qatlamni 1000 sinfdan bizning 5 sinfga almashtiramiz

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 154MB/s]


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # GPU bo'lsa 'cuda', bo'lmasa 'cpu' qurilmasini tanlaymiz
print('device', device)                                                 # Qaysi qurilma tanlanganini chiqaramiz → device cuda
model = model.to(device)                                                # Modelni tanlangan qurilmaga ko'chiramiz

device cuda


In [12]:
criterion = nn.CrossEntropyLoss()   # Loss Function | '70%' burger, '30%' pilaf   → klassifikatsiya uchun xato funksiyasi
optimizer = optim.Adam(model.parameters(), lr=0.001)   # weight   → modelning barcha og'irliklarini Adam usuli bilan yangilaydi
torch.backends.cudnn.benchmark = True   # Benchmark Setting | Trick | 10%-20%   → GPU uchun eng tez hisoblash usulini avtomatik tanlaydi